# Data Understanding and Preparation

#### Problem #1 ####
The full 311 Dataset contains 22,542,090 entries and it is growing daily. Github has file size limits
Therefore, we can't upload the entire dataset, it has to be filtered

We would like to take the most recent data, since it better reflects current City operations (staffing, policy, procedures)
Considered intervals: last 5 years, last 2 years, last year

**Decision:** take data for the last year (since sept 2025) - large enough for solid model training, small enough to be manageable to download and clean.
A full 12-month window also ensures the data captures a complete seasonal cycle (e.g., heat complaints peak in winter, noise complaints in summer), avoiding gaps from an incomplete calendar year.

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


In [ ]:
if not os.path.isdir('dataset') and os.path.isdir('../dataset'):
    os.chdir('..')

filename = os.path.join('dataset', '311_Service_Requests_from_2020_to_Present.zip')

In [25]:
df = pd.read_csv(filename, low_memory=False)

##
## Initial Exploration

In [26]:
print(f"Number of Rows: {df.shape[0]}")
print(f"Number of Columns: {df.shape[1]}")
df.head()

Number of Rows: 3940809
Number of Columns: 44


,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Vehicle Type,Taxi Company Borough,Taxi Pick Up Location,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location
0,70471747,09/20/2026 01:51:01 AM,NaN,NYPD,New York City Police Department,Illegal Parking,Blocked Hydrant,NaN,Street/Sidewalk,10016,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.744553,-73.977398,POINT (-73.977397686134 40.744552551425)
1,70474903,09/20/2026 01:50:51 AM,NaN,DEP,Department of Environmental Protection,Water Maintenance,Water Main Break,NaN,Street,11238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.672907,-73.959725,POINT (-73.959725084153 40.672906659623)
2,70470200,09/20/2026 01:50:51 AM,NaN,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11373,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.737271,-73.879943,POINT (-73.879942579006 40.737270748449)
3,70475185,09/20/2026 01:50:45 AM,NaN,NYPD,New York City Police Department,Noise - Vehicle,Car/Truck Music,NaN,Street/Sidewalk,11220,...,Car,NaN,NaN,NaN,NaN,NaN,NaN,40.641947,-73.999272,POINT (-73.999272130506 40.641947092776)
4,70475938,09/20/2026 01:50:40 AM,NaN,NYPD,New York City Police Department,Blocked Driveway,No Access,NaN,Street/Sidewalk,11433,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,40.689879,-73.791066,POINT (-73.791066478538 40.689878873276)


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3940809 entries, 0 to 3940808
Data columns (total 44 columns):
 #   Column                                Dtype  
---  ------                                -----  
 0   Unique Key                            int64  
 1   Created Date                          object 
 2   Closed Date                           object 
 3   Agency                                object 
 4   Agency Name                           object 
 5   Problem (formerly Complaint Type)     object 
 6   Problem Detail (formerly Descriptor)  object 
 7   Additional Details                    object 
 8   Location Type                         object 
 9   Incident Zip                          object 
 10  Incident Address                      object 
 11  Street Name                           object 
 12  Cross Street 1                        object 
 13  Cross Street 2                        object 
 14  Intersection Street 1                 object 
 15  Intersection St

## Dates & Resolution Time - Label Definition

The dataset does not include a precomputed resolution time - only `Created Date` and `Closed Date` timestamps for each complaint.

To compute resolution time, we took the difference between `Closed Date` and `Created Date`, converted into a continuous numeric value (hours and days), which serves as our model's target variable.

**Steps:**
1. Convert `Created Date` and `Closed Date` to datetime format
2. Compute `resolution_time = Closed Date - Created Date`
3. Convert to `resolution_days` and `resolution_hours`
4. Drop rows with no `Closed Date` - these are still-open complaints, therefore, resolution time cannot be computed
5. Remove rows with negative resolution time (data entry errors, where `Closed Date` precedes `Created Date`) if any

In [ ]:
df['Created Date'] = pd.to_datetime(df['Created Date'], errors='coerce')
df['Closed Date'] = pd.to_datetime(df['Closed Date'], errors='coerce')

missing_closed_date = df['Closed Date'].isna().sum()
print(f"Number of missing 'Closed Date' values: {missing_closed_date}")
df = df.dropna(subset=['Closed Date'])

df['resolution_min'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 60
df['resolution_hours'] = (df['Closed Date'] - df['Created Date']).dt.total_seconds() / 3600
df['resolution_days'] = df['resolution_hours'] / 24

negative_resolution = (df['resolution_hours'] < 0).sum()
print(f"Number of rows with negative resolution time: {negative_resolution}")

df = df[df['resolution_hours'] >= 0]

Number of missing 'Closed Date' values: 0
Number of rows with negative resolution time: 0


In [31]:
df.head()

,Unique Key,Created Date,Closed Date,Agency,Agency Name,Problem (formerly Complaint Type),Problem Detail (formerly Descriptor),Additional Details,Location Type,Incident Zip,...,Bridge Highway Name,Bridge Highway Direction,Road Ramp,Bridge Highway Segment,Latitude,Longitude,Location,resolution_hours,resolution_days,resolution_min
75,70475296,2026-09-20 01:39:36,2026-09-20 01:47:32,NYPD,New York City Police Department,Noise - Residential,Loud Music/Party,NaN,Residential Building/House,11208,...,NaN,NaN,NaN,NaN,40.666362,-73.876732,POINT (-73.876732316369 40.666361759266),0.132222,0.005509,7.933333
90,70471164,2026-09-20 01:37:27,2026-09-20 01:47:26,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Music/Party,NaN,Street/Sidewalk,11222,...,NaN,NaN,NaN,NaN,40.730319,-73.960665,POINT (-73.960664569783 40.73031925345),0.166389,0.006933,9.983333
95,70470378,2026-09-20 01:36:25,2026-09-20 01:45:23,NYPD,New York City Police Department,Noise - Park,Loud Music/Party,NaN,Park/Playground,10033,...,NaN,NaN,NaN,NaN,40.843501,-73.945906,POINT (-73.945905791727 40.843500551231),0.149444,0.006227,8.966667
99,70478136,2026-09-20 01:35:40,2026-09-20 01:49:29,NYPD,New York City Police Department,Noise - Street/Sidewalk,Loud Talking,NaN,Street/Sidewalk,11238,...,NaN,NaN,NaN,NaN,40.676053,-73.961223,POINT (-73.961222929757 40.676052686722),0.230278,0.009595,13.816667
106,70478077,2026-09-20 01:34:30,2026-09-20 01:44:49,NYPD,New York City Police Department,Noise - Commercial,Banging/Pounding,NaN,Club/Bar/Restaurant,10002,...,NaN,NaN,NaN,NaN,40.718013,-73.989896,POINT (-73.989895597843 40.718012546099),0.171944,0.007164,10.316667


#### Problem #2 ####
What time unite should be use for model? Minutes, hours or days.

In [33]:
#How many resolved in under 10 minute?
under_10min = (df['resolution_min'] < 10).sum()
print(f"Resolved in under 10 minutes: {under_10min}, or {under_10min/len(df)*100:.1f}%")

# How many resolved in under 1 hour?
under_1hr = (df['resolution_hours'] < 1).sum()
print(f"Resolved in under 1 hour: {under_1hr}, or {under_1hr/len(df)*100:.1f}%")

# How many resolved in under 1 day?
under_1day = (df['resolution_days'] < 1).sum()
print(f"Resolved in under 1 day: {under_1day}, or {under_1day/len(df)*100:.1f}%")

# How many took more than a week?
over_1week = (df['resolution_days'] > 7).sum()
print(f"Took more than 1 week: {over_1week}, or {over_1week/len(df)*100:.1f}%")

# How many took more than a month?
over_1month = (df['resolution_days'] > 30).sum()
print(f"Took more than 1 month: {over_1month}, or {over_1month/len(df)*100:.1f}%")

Resolved in under 10 minutes: 186439, or 5.0%
Resolved in under 1 hour: 868230, or 23.1%
Resolved in under 1 day: 2290812, or 60.9%
Took more than 1 week: 600382, or 16.0%
Took more than 1 month: 222283, or 5.9%


### Decision: ### 
***Model target:*** vast majority of the complaints (60%) were resolved within a day, therefore model target is `resolution_hours` - the model predicts an exact, continuous value (hours), not a bucket or category. 

***Why not classification/buckets:*** Bucketing resolution time into ranges (e.g., "1-7 days") was considered, but rejected as the model's primary output - a bucket is functionally similar to the City's existing SLA system (a static range/commitment), which undermines this project's core value proposition: giving residents a *realistic, data-grounded* estimate that generic SLAs and AI/search tools can't provide.

The trained model always returns one exact number. The AI agent layer sits on top of this and:
- Displays the **exact predicted time** as the default answer, converting hours into the most natural unit for the user (minutes, hours, or days depending on scale - e.g., "23 minutes" vs. "4.6 days")
- Provides a range computed as the 10th–90th percentile of historical resolution times for similar complaints (same complaint type/borough), not a fixed bucket

## Features exploration

In [36]:
columns_list = df.columns.tolist()
print(f"Dataset contains following columns: {columns_list}")

categorical_columns = df.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns: {categorical_columns}")

Dataset contains following columns: ['Unique Key', 'Created Date', 'Closed Date', 'Agency', 'Agency Name', 'Problem (formerly Complaint Type)', 'Problem Detail (formerly Descriptor)', 'Additional Details', 'Location Type', 'Incident Zip', 'Incident Address', 'Street Name', 'Cross Street 1', 'Cross Street 2', 'Intersection Street 1', 'Intersection Street 2', 'Address Type', 'City', 'Landmark', 'Facility Type', 'Status', 'Due Date', 'Resolution Description', 'Resolution Action Updated Date', 'Community Board', 'Council District', 'Police Precinct', 'BBL', 'Borough', 'X Coordinate (State Plane)', 'Y Coordinate (State Plane)', 'Open Data Channel Type', 'Park Facility Name', 'Park Borough', 'Vehicle Type', 'Taxi Company Borough', 'Taxi Pick Up Location', 'Bridge Highway Name', 'Bridge Highway Direction', 'Road Ramp', 'Bridge Highway Segment', 'Latitude', 'Longitude', 'Location', 'resolution_hours', 'resolution_days', 'resolution_min']
Categorical columns: ['Agency', 'Agency Name', 'Problem 

## Initial Feature List

Based on the available columns, the following features were selected for exploration
**Categorical features:**
- `Problem (formerly Complaint Type)`
- `Problem Detail (formerly Descriptor)`
- `Agency`
- `Borough`
- `Location Type`
- `Open Data Channel Type`
- `Community Board`
- `Police Precinct`


- `Created Date`


**Excluded** (only known after a complaint is resolved, not at filing time):
- `Closed Date`
- `Status`
- `Resolution Description`
- `Resolution Action Updated Date`

This list will be refined based on missingness, cardinality, and observed relationship to resolution time during exploration.

#### Categorical Features

In [41]:
features_to_explore = [
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Agency',
    'Borough',
    'Location Type',
    'Open Data Channel Type',
    'Community Board',
    'Police Precinct',
    'Due Date',
    'created_hour',
    'created_dayofweek',
    'is_weekend',
    'created_month',
]

# Categorical features
categorical_features = [
    'Problem (formerly Complaint Type)',
    'Problem Detail (formerly Descriptor)',
    'Agency',
    'Borough',
    'Location Type',
    'Open Data Channel Type',
    'Community Board',
    'Police Precinct',
]


for col in categorical_features:
    print(f"\n--- {col} ---")
    print(df[col].value_counts().head(20))
    print(f"Number of unique values in {col}: {df[col].nunique()}")


--- Problem (formerly Complaint Type) ---
Problem (formerly Complaint Type)
Illegal Parking             608319
Noise - Residential         433900
HEAT/HOT WATER              345568
Blocked Driveway            193197
Noise - Street/Sidewalk     175293
UNSANITARY CONDITION        125447
Street Condition            110083
PLUMBING                     78145
Abandoned Vehicle            70784
Water System                 69755
Dirty Condition              66547
PAINT/PLASTER                64370
Snow or Ice                  63828
Noise - Commercial           60070
Noise                        58140
Traffic Signal Condition     56922
Noise - Vehicle              51040
DOOR/WINDOW                  49401
Encampment                   46410
WATER LEAK                   44485
Name: count, dtype: int64
Number of unique values in Problem (formerly Complaint Type): 191

--- Problem Detail (formerly Descriptor) ---
Problem Detail (formerly Descriptor)
Loud Music/Party                  444380
ENTIRE 

***Findings***
- **Complaint Type** has 191 unique values, heavily concentrated at the top - Illegal Parking, Noise - Residential, and HEAT/HOT WATER alone account for a large share of all complaints. Rare categories will need to be consolidated into "Other" before modeling.
- **Complaint Type** and **Location Type** values have inconsistent casing (e.g., "HEAT/HOT WATER" vs. "Street Condition"), which needs standardizing so similar categories aren't treated as distinct due to formatting alone.
- **Problem Detail (Descriptor)** has 1,008 unique values — too granular to use directly as a model feature without significant consolidation.
- **Agency** is clean with only 16 values; NYPD (1.7M) and HPD (848K) dominate by volume. Strong, low-effort candidate feature.
- **Borough** is clean with 6 values (5 boroughs + "Unspecified," 2,892 rows) — need to decide how to handle the "Unspecified" category.
- **Location Type** has 152 values, including clear duplicates representing the same real-world category (e.g., "Residential Building/House," "RESIDENTIAL BUILDING," "3+ Family Apt. Building," and "3+ Family Apartment Building" all appear to refer to the same thing). These need manual merging, not just rare-category consolidation.
- **Open Data Channel Type** is clean with 5 values; a notable share (308K) are "UNKNOWN" — worth checking whether this correlates with resolution time.
- **Community Board** and **Police Precinct** are high-cardinality (77–78 values each) and largely redundant with Borough at a coarser level. Plan: start modeling with Borough only; treat these as a possible later enhancement.

#### Time-based features

In [42]:
df['created_hour'] = df['Created Date'].dt.hour
df['created_dayofweek'] = df['Created Date'].dt.dayofweek
df['created_month'] = df['Created Date'].dt.month
df['is_weekend'] = df['created_dayofweek'].isin([5, 6])


time_features = [
    'created_hour',
    'created_dayofweek',
    'is_weekend',
    'created_month',
]

In [43]:
print("Hour of day distribution:")
print(df['created_hour'].value_counts().sort_index())

print("\nDay of week distribution:")
print(df['created_dayofweek'].value_counts().sort_index())

print("\nWeekend vs weekday:")
print(df['is_weekend'].value_counts())

print("\nMonth distribution:")
print(df['created_month'].value_counts().sort_index())

Hour of day distribution:
created_hour
0     131342
1      88177
2      60196
3      47359
4      45006
5      53489
6      83919
7     145941
8     198625
9     222308
10    223623
11    220091
12    212192
13    204877
14    201728
15    197214
16    190510
17    181649
18    174650
19    172856
20    174103
21    185309
22    187555
23    157200
Name: count, dtype: int64

Day of week distribution:
created_dayofweek
0    576288
1    576945
2    542433
3    532458
4    542281
5    489491
6    500023
Name: count, dtype: int64

Weekend vs weekday:
is_weekend
False    2770405
True      989514
Name: count, dtype: int64

Month distribution:
created_month
1     340115
2     323870
3     330887
4     291557
5     318534
6     319051
7     322751
8     300785
9     255119
10    329596
11    300436
12    327218
Name: count, dtype: int64


In [44]:
print("Median resolution_hours by hour of day:")
print(df.groupby('created_hour')['resolution_hours'].median())

print("\nMedian resolution_hours by day of week:")
print(df.groupby('created_dayofweek')['resolution_hours'].median())

print("\nMedian resolution_hours: weekend vs weekday:")
print(df.groupby('is_weekend')['resolution_hours'].median())

print("\nMedian resolution_hours by month:")
print(df.groupby('created_month')['resolution_hours'].median())

Median resolution_hours by hour of day:
created_hour
0      1.954444
1      2.193056
2      2.315139
3      2.471111
4      2.853750
5      3.616667
6      4.548056
7      6.600000
8     11.505833
9     22.688472
10    26.381667
11    25.427222
12    24.716667
13    23.483056
14    22.219444
15    21.037500
16    18.708611
17    11.693056
18     7.005833
19     5.233750
20     3.939444
21     3.120278
22     2.539444
23     2.271389
Name: resolution_hours, dtype: float64

Median resolution_hours by day of week:
created_dayofweek
0    12.308056
1    12.030000
2     9.816667
3     9.033333
4     7.407778
5     3.568056
6     3.754167
Name: resolution_hours, dtype: float64

Median resolution_hours: weekend vs weekday:
is_weekend
False    9.931111
True     3.655833
Name: resolution_hours, dtype: float64

Median resolution_hours by month:
created_month
1     25.885000
2     16.566667
3      8.301111
4      6.478611
5      5.170556
6      5.662222
7      5.341944
8      5.299444
9      2.965

***Findings***

- **Hour of day** shows a strong pattern: resolution time is lowest overnight (~2 hours around midnight–4am) and peaks sharply during business hours (22–26 hours between 9am–1pm), declining again through the evening.
- **Day of week** shows weekday complaints take roughly 2–3x longer to resolve than weekend complaints (e.g., Monday: ~12.3 hours vs. Saturday: ~3.6 hours).
- **Weekend vs. weekday** confirms this cleanly as a simple binary split: ~9.9 hours (weekday) vs. ~3.7 hours (weekend).
- **Month** shows a seasonal pattern, with January notably highest (~25.9 hours) and September lowest (~3.0 hours), generally declining from winter into fall before rising again toward year-end.
- All four time-based features show meaningful variation in resolution time and are strong candidates for the model.
- **Caveat to investigate further:** these time patterns may be partly driven by *which* complaint types tend to get filed at different times (e.g., late-night/weekend complaints skewing toward faster-resolving categories like noise), rather than time itself directly affecting resolution speed. To be checked via cross-tabulation of complaint type against hour/weekend before finalizing feature interpretation.
- September's low resolution time coincides with its lower complaint volume (seen in the earlier distribution check) — worth confirming the dataset's date range fully covers September, to rule out a partial-month artifact.

## Next Steps

### EDA (remaining)
- [ ] Visualize the resolution time distribution (histogram, log scale) to confirm the skew and support the log-transform decision
- [ ] Boxplot of resolution time by top complaint types, to inspect variance/outliers per category
- [ ] Check whether time-based feature patterns (hour, weekend) are driven by complaint type mix (cross-tab complaint type against hour-of-day / weekend, to confirm whether time features add independent signal)

### Data Cleaning
- [ ] Full missing-values check across all columns (`df.isna().sum()`)
- [ ] Check for duplicate rows (`df.duplicated().sum()`)
- [ ] Standardize categorical casing (Complaint Type, Location Type)
- [ ] Merge duplicate Location Type values (e.g., "Residential Building/House" vs. "RESIDENTIAL BUILDING")
- [ ] Consolidate rare categories into "Other" (Complaint Type, Location Type)
- [ ] Resolve `Due Date` missingness / decide its role (feature vs. evaluation benchmark)
- [ ] Handle remaining missing values per column (fill vs. drop)

### Feature Engineering
- [ ] Finalize feature set
- [ ] Encode categorical variables

### Modeling
- [ ] Baseline model (median resolution time by complaint type + agency)